In [18]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')
pd.set_option("display.max_columns", 500)

EVENTS_DIR = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_enriched"
HOURLY_DIR = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_hourly_data"
VAULTS_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/common/vaults_list.csv"
SUPPLIERS_SHARE_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_suppliers_share.csv"
BORROWERS_SHARE_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_borrowers_share.csv"
SPIKES_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/all_spikes_dataset.csv"

MARKET_LIST = [
    'eth_rlp_usdc',
]


def load_spikes(spikes_path):
    df = pd.read_csv(spikes_path)
    df['spike_trigger_datetime'] = pd.to_datetime(df['spike_trigger_datetime'])
    df['spike_recovery_datetime'] = pd.to_datetime(df['spike_recovery_datetime'])
    return df


def filter_markets(spikes_df, market_list):
    return spikes_df[spikes_df['market_name'].isin(market_list)].copy()


def generate_hourly_skeleton(spike_rows):
    skeleton = []
    for idx, row in spike_rows.iterrows():
        start = row['spike_trigger_datetime'].floor('h')
        end = row['spike_recovery_datetime'].ceil('h')
        if pd.isna(end) or pd.isna(start):
            continue
        hours = pd.date_range(start=start, end=end, freq='h')
        for i, hour_start in enumerate(hours):
            skeleton.append({
                'spike_id': idx,
                'market_name': row['market_name'],
                'hour_start_timestamp': int(hour_start.timestamp()),
                'tau': i,
                'spike_trigger_datetime': row['spike_trigger_datetime'],
                'spike_recovery_datetime': row['spike_recovery_datetime'],
                'peak_utilization': row['peak_utilization'],
                'utilization_before': row['utilization_before']
            })
    return pd.DataFrame(skeleton)


def load_hourly_data(market_name, hourly_dir):
    file_path = Path(hourly_dir) / f"{market_name}.csv"
    if not file_path.exists():
        return pd.DataFrame()
    df = pd.read_csv(file_path)
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['timestamp'] = df['timestamp'].astype(int)
    return df


def attach_hourly_features(skeleton, hourly_df):
    skeleton = skeleton.sort_values(['market_name', 'hour_start_timestamp'])
    hourly_df = hourly_df.sort_values('timestamp')
    merged = pd.merge_asof(
        skeleton,
        hourly_df[['timestamp', 'total_supply', 'total_borrow', 'utilization',
                   'borrow_rate', 'supply_rate', 'collateral_price', 'loan_asset_price']],
        left_on='hour_start_timestamp',
        right_on='timestamp',
        direction='backward',
        by='market_name' if 'market_name' in hourly_df.columns else None
    )
    if 'market_name' not in hourly_df.columns:
        merged = merged.drop(columns=['timestamp'])
    else:
        merged = merged.drop(columns=['timestamp', 'market_name_y']).rename(columns={'market_name_x': 'market_name'})
    return merged


def load_events_data(market_name, events_dir):
    file_path = Path(events_dir) / f"{market_name}.csv"
    if not file_path.exists():
        return pd.DataFrame()
    df = pd.read_csv(file_path)
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['timestamp'] = df['timestamp'].astype(int)
    return df


def compute_hourly_inflow(events_df, hour_start, hour_end):
    mask = (events_df['timestamp'] >= hour_start) & (events_df['timestamp'] < hour_end)
    mask &= (events_df['type'] == 'MarketSupply')
    hour_events = events_df[mask]
    if hour_events.empty:
        return 0.0, 0.0, 0.0
    total = hour_events['assets_usd'].sum()
    vault = hour_events[hour_events['vault_flg'] == True]['assets_usd'].sum()
    retail = hour_events[hour_events['vault_flg'] == False]['assets_usd'].sum()
    return total, vault, retail


def load_share_data(share_path, market_name, side):
    df = pd.read_csv(share_path)
    df = df[(df['market'] == market_name) & (df['side'] == side)].copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['timestamp'] = df['timestamp'].astype(int)
    return df.sort_values('timestamp')


def load_vaults(vaults_path):
    return pd.read_csv(vaults_path)


def attach_concentration_features(skeleton, suppliers_share_df, borrowers_share_df, vaults_df):
    skeleton = skeleton.sort_values(['market_name', 'hour_start_timestamp'])
    suppliers_share_df = suppliers_share_df.sort_values('timestamp')
    borrowers_share_df = borrowers_share_df.sort_values('timestamp')

    merged = pd.merge_asof(
        skeleton,
        suppliers_share_df[['timestamp', 'hhi', 'n_active']],
        left_on='hour_start_timestamp',
        right_on='timestamp',
        direction='backward'
    ).rename(columns={'hhi': 'supplier_hhi', 'n_active': 'active_suppliers'})
    merged = merged.drop(columns=['timestamp'])

    merged = pd.merge_asof(
        merged,
        borrowers_share_df[['timestamp', 'hhi', 'n_active']],
        left_on='hour_start_timestamp',
        right_on='timestamp',
        direction='backward'
    ).rename(columns={'hhi': 'borrower_hhi', 'n_active': 'active_borrowers'})
    merged = merged.drop(columns=['timestamp'])

    vault_addresses = set(vaults_df['address'])
    vault_share = []
    for _, row in merged.iterrows():
        market = row['market_name']
        ts = row['hour_start_timestamp']
        sup = suppliers_share_df[(suppliers_share_df['timestamp'] <= ts)]
        if sup.empty:
            vault_share.append(np.nan)
            continue
        latest_ts = sup['timestamp'].max()
        latest_sup = sup[sup['timestamp'] == latest_ts]
        vault_share_sum = latest_sup[latest_sup['user_address'].isin(vault_addresses)]['share'].sum()
        vault_share.append(vault_share_sum)
    merged['vault_share_of_supply'] = vault_share

    return merged


def engineer_features(panel_df):
    df = panel_df.copy()
    df = df.sort_values(['market_name', 'hour_start_timestamp'])
    df['prev_borrow_rate'] = df.groupby('market_name')['borrow_rate'].shift(1)
    df['rate_acceleration'] = (df['borrow_rate'] - df['prev_borrow_rate']) / df['prev_borrow_rate'].replace(0, np.nan)
    df['rate_acceleration'] = df['rate_acceleration'].fillna(0)
    df['distance_to_kink'] = np.maximum(0, df['utilization'] - 0.90)
    df['is_above_kink'] = (df['utilization'] >= 0.90).astype(int)
    df['vault_dominance_flag'] = (df['vault_share_of_supply'] >= 0.5).astype(int)
    df['market_tvl_usd'] = df['total_supply'] * df['collateral_price']
    df['tau_squared'] = df['tau'] ** 2
    df.drop(columns=['prev_borrow_rate'], inplace=True)
    return df


def build_panel(market_list, spikes_path, hourly_dir, events_dir, suppliers_path, borrowers_path, vaults_path):
    spikes_df = load_spikes(spikes_path)
    filtered_spikes = filter_markets(spikes_df, market_list)
    if filtered_spikes.empty:
        return pd.DataFrame()

    skeleton = generate_hourly_skeleton(filtered_spikes)

    all_panels = []
    for market in market_list:
        market_skeleton = skeleton[skeleton['market_name'] == market].copy()
        if market_skeleton.empty:
            continue

        hourly_df = load_hourly_data(market, hourly_dir)
        if hourly_df.empty:
            continue
        market_skeleton = attach_hourly_features(market_skeleton, hourly_df)

        events_df = load_events_data(market, events_dir)
        inflows = []
        for _, row in market_skeleton.iterrows():
            hour_start = row['hour_start_timestamp']
            hour_end = hour_start + 3600 * 3
            total, vault, retail = compute_hourly_inflow(events_df, hour_start, hour_end)
            inflows.append((total, vault, retail))
        market_skeleton[['delta_supply_lenders_usd', 'delta_supply_vault_usd', 'delta_supply_retail_usd']] = pd.DataFrame(inflows, index=market_skeleton.index)

        suppliers_share = load_share_data(suppliers_path, market, 'supply')
        borrowers_share = load_share_data(borrowers_path, market, 'borrow')
        vaults_df = load_vaults(vaults_path)

        market_skeleton = attach_concentration_features(market_skeleton, suppliers_share, borrowers_share, vaults_df)
        market_skeleton = engineer_features(market_skeleton)

        all_panels.append(market_skeleton)

    if not all_panels:
        return pd.DataFrame()
    final_panel = pd.concat(all_panels, ignore_index=True)

    final_panel = final_panel[final_panel['total_supply'] > 0]
    final_panel = final_panel.groupby('spike_id').filter(lambda x: len(x) >= 2)

    cols = [
        'spike_id', 'market_name', 'hour_start_timestamp', 'tau',
        'delta_supply_lenders_usd', 'delta_supply_vault_usd', 'delta_supply_retail_usd',
        'utilization', 'borrow_rate', 'supply_rate', 'total_supply', 'total_borrow',
        'supplier_hhi', 'active_suppliers', 'vault_share_of_supply',
        'borrower_hhi', 'active_borrowers',
        'rate_acceleration', 'distance_to_kink', 'is_above_kink',
        'vault_dominance_flag', 'market_tvl_usd', 'tau_squared'
    ]
    final_panel = final_panel[cols]

    return final_panel


if __name__ == "__main__":
    panel = build_panel(
        MARKET_LIST,
        SPIKES_PATH,
        HOURLY_DIR,
        EVENTS_DIR,
        SUPPLIERS_SHARE_PATH,
        BORROWERS_SHARE_PATH,
        VAULTS_PATH
    )
    print(f"Panel shape: {panel.shape}")
    display(panel.head())

Panel shape: (1077, 23)


,spike_id,market_name,hour_start_timestamp,tau,delta_supply_lenders_usd,delta_supply_vault_usd,delta_supply_retail_usd,utilization,borrow_rate,supply_rate,total_supply,total_borrow,supplier_hhi,active_suppliers,vault_share_of_supply,borrower_hhi,active_borrowers,rate_acceleration,distance_to_kink,is_above_kink,vault_dominance_flag,market_tvl_usd,tau_squared
0,1060,eth_rlp_usdc,1753372800,0,1.407964e+07,1.407964e+07,0.000000,0.910369,0.115955,0.105571,1.112715e+08,1.012980e+08,0.060345,220,0.0,0.066960,127,0.000000,0.010369,1,0,1.357512e+08,0
1,1060,eth_rlp_usdc,1753376400,1,9.325528e+06,9.300532e+06,24996.008523,0.919042,0.139060,0.127825,1.175336e+08,1.080183e+08,0.062616,221,0.0,0.067378,128,0.199261,0.019042,1,0,1.433910e+08,1
2,1061,eth_rlp_usdc,1753387200,0,1.878508e+07,1.870237e+07,82704.586872,0.919596,0.141026,0.129697,1.175350e+08,1.080848e+08,0.062277,222,0.0,0.067297,129,0.014140,0.019596,1,0,1.433927e+08,0
3,1061,eth_rlp_usdc,1753390800,1,1.480963e+07,1.467068e+07,138952.654897,0.919520,0.141024,0.129687,1.175475e+08,1.080872e+08,0.062277,222,0.0,0.067297,129,-0.000013,0.019520,1,0,1.434079e+08,1
4,1062,eth_rlp_usdc,1753484400,0,7.955640e+06,7.527817e+06,427822.482788,0.916492,0.136047,0.124749,1.190693e+08,1.091261e+08,0.063384,225,0.0,0.065891,132,-0.035291,0.016492,1,0,1.452646e+08,0


In [19]:
panel.head(50)

,spike_id,market_name,hour_start_timestamp,tau,delta_supply_lenders_usd,delta_supply_vault_usd,delta_supply_retail_usd,utilization,borrow_rate,supply_rate,total_supply,total_borrow,supplier_hhi,active_suppliers,vault_share_of_supply,borrower_hhi,active_borrowers,rate_acceleration,distance_to_kink,is_above_kink,vault_dominance_flag,market_tvl_usd,tau_squared
0,1060,eth_rlp_usdc,1753372800,0,1.407964e+07,1.407964e+07,0.000000e+00,0.910369,0.115955,0.105571,1.112715e+08,1.012980e+08,0.060345,220,0.0,0.066960,127,0.000000,0.010369,1,0,1.357512e+08,0
1,1060,eth_rlp_usdc,1753376400,1,9.325528e+06,9.300532e+06,2.499601e+04,0.919042,0.139060,0.127825,1.175336e+08,1.080183e+08,0.062616,221,0.0,0.067378,128,0.199261,0.019042,1,0,1.433910e+08,1
2,1061,eth_rlp_usdc,1753387200,0,1.878508e+07,1.870237e+07,8.270459e+04,0.919596,0.141026,0.129697,1.175350e+08,1.080848e+08,0.062277,222,0.0,0.067297,129,0.014140,0.019596,1,0,1.433927e+08,0
3,1061,eth_rlp_usdc,1753390800,1,1.480963e+07,1.467068e+07,1.389527e+05,0.919520,0.141024,0.129687,1.175475e+08,1.080872e+08,0.062277,222,0.0,0.067297,129,-0.000013,0.019520,1,0,1.434079e+08,1
4,1062,eth_rlp_usdc,1753484400,0,7.955640e+06,7.527817e+06,4.278225e+05,0.916492,0.136047,0.124749,1.190693e+08,1.091261e+08,0.063384,225,0.0,0.065891,132,-0.035291,0.016492,1,0,1.452646e+08,0
5,1062,eth_rlp_usdc,1753488000,1,3.825466e+06,3.280547e+06,5.449188e+05,0.918374,0.141321,0.129822,1.235846e+08,1.134969e+08,0.061813,225,0.0,0.063130,132,0.038760,0.018374,1,0,1.507732e+08,1
6,1063,eth_rlp_usdc,1753513200,0,9.210780e+06,8.297134e+06,9.136456e+05,0.907790,0.113032,0.102657,1.268013e+08,1.151089e+08,0.061319,225,0.0,0.062634,132,-0.200173,0.007790,1,0,1.546975e+08,0
7,1063,eth_rlp_usdc,1753516800,1,3.281059e+06,2.471462e+06,8.095970e+05,0.907831,0.113184,0.102798,1.267969e+08,1.151101e+08,0.061319,225,0.0,0.062634,132,0.001342,0.007831,1,0,1.546923e+08,1
8,1064,eth_rlp_usdc,1753588800,0,7.527382e+06,7.456836e+06,7.054679e+04,0.914794,0.132977,0.121715,1.248404e+08,1.142033e+08,0.061619,225,0.0,0.062854,133,0.174877,0.014794,1,0,1.523053e+08,0
9,1064,eth_rlp_usdc,1753592400,1,1.696270e+07,1.689787e+07,6.482570e+04,0.917802,0.141359,0.129787,1.289090e+08,1.183129e+08,0.061547,225,0.0,0.063026,133,0.063033,0.017802,1,0,1.572689e+08,1
